# NISAR Mangrove Change Detection
## Notebook 02: HDF5 Structure Inspection & Preprocessing Feasibility (Phase 3A)

### Objective
This notebook performs **Phase 3A feasibility inspection** without performing bulk downloads or premature change classification:
1. **Product Selection & Ephemeris Quality Audit**: Review Track 019 and Track 170 candidates, specifically evaluating the September 17 and September 19 Forecast ephemeris products against archive holdings.
2. **Multi-Temporal Geometry Consistency**: Evaluate look geometry and incidence angle variations across the 12-day Track 019 repeat acquisitions over Gosaba.
3. **HDF5 Dataset Architecture**: Inspect the internal HDF5 hierarchy (`/science/LSAR/GCOV/grids/frequencyA`), identifying covariance layers, coordinate vectors, and quality masks according to JPL D-102274.
4. **Gosaba Subsetting Feasibility**: Map geographic coordinates ($88.70^\circ - 89.00^\circ\text{ E}, 21.80^\circ - 22.10^\circ\text{ N}$) to raster row/column slice ranges in UTM Zone 45N.
5. **Remote Access Feasibility**: Test HTTP range request behavior on NASA Earthdata Cloud endpoints.
6. **Spatial Resolution & Grid Harmonization**: Define master grid resampling specifications to harmonize L-band (~24 cm) and S-band (~9.3 cm) observations.

In [ ]:
# Step 1: Environment Setup
import sys
import os
import json
import math
from pathlib import Path

# Auto-install dependencies if in Google Colab
if 'google.colab' in sys.modules:
    !pip install -q h5py numpy pandas matplotlib folium
    print("[INFO] Google Colab environment configured.")
else:
    print("[INFO] Standard environment configured.")

import pandas as pd
import numpy as np

print("[OK] Libraries loaded.")

In [ ]:
# Step 2: Product Quality & Ephemeris Audit
# Load the audited product quality catalog
audit_file = Path("../data/processed/product_quality_audit.csv")

if audit_file.exists():
    df_audit = pd.read_csv(audit_file)
    print(f"[OK] Loaded {len(df_audit)} audited granules.")
    
    # Filter to candidate scenes
    selected = df_audit[df_audit["selection_status"] == "SELECTED"]
    print(f"\n--- Primary Candidate Scenes ({len(selected)}) ---")
    display(selected[[
        "acquisition_date_utc", "start_time_utc", "track", "flight_direction",
        "frame", "product_maturity", "ephemeris_accuracy", "counter", "estimated_incidence_angle_at_gosaba"
    ]])
else:
    print("[WARNING] product_quality_audit.csv not found.")

### Step 3: Investigation of September 17 & September 19 Forecast Products

An exhaustive query of the NASA ASF archive was executed covering September 16 to September 20, 2026.

#### Key Findings:
1. **Archive Status**: Only `_F_F_J_001` versions exist in the ASF archive for Track 170 on Sep 17 and Track 019 on Sep 19. No Near-Real-Time (`_N_`) or Precise (`_P_`) reprocessed runs exist.
2. **Selection Decision**:
   - On September 17, `NISAR_L2_PR_GCOV_030_170_A_013_..._F_F_J_001` is selected because it represents standard Provisional science processing, superseding the Urgent Response (`UR`) product.
   - On September 19, `NISAR_L2_PR_GCOV_031_019_D_077_..._F_F_J_001` is selected as the sole Level-2 GCOV product covering Gosaba for Cycle 31.
3. **Scientific Uncertainty**: Forecast ephemeris introduces potential geometric geolocation uncertainty ($\sim 10 - 30\text{ m}$) compared to definitive precise ephemeris ($< 5\text{ m}$). This requires empirical spatial co-registration during preprocessing.

In [ ]:
# Step 4: Multi-Temporal Geometry Consistency (Track 019 Quad-Pol Series)
# Comparing footprints of the 4 repeat passes: Aug 26, Sep 07, Sep 19, Oct 01, 2026

footprints_t19 = [
    {"cycle": 29, "date": "2026-08-26", "west_lon": 87.08989, "east_lon": 90.10617, "north_lat": 24.14092, "south_lat": 21.47369},
    {"cycle": 30, "date": "2026-09-07", "west_lon": 87.16745, "east_lon": 90.12159, "north_lat": 24.17644, "south_lat": 21.46917},
    {"cycle": 31, "date": "2026-09-19", "west_lon": 87.08945, "east_lon": 90.12378, "north_lat": 24.19984, "south_lat": 21.47321},
    {"cycle": 32, "date": "2026-10-01", "west_lon": 87.16913, "east_lon": 90.12359, "north_lat": 24.18308, "south_lat": 21.47577}
]

df_geo = pd.DataFrame(footprints_t19)
print("Track 019 Swath Boundary Variations Across Cycles:")
display(df_geo)

# Analysis:
print("\n[GEOMETRY AUDIT FINDINGS]")
print(f"Max Western Boundary Shift: {df_geo['west_lon'].max() - df_geo['west_lon'].min():.5f} deg longitude (~8.2 km ground distance)")
print("Nominal Local Incidence Angle at Gosaba (88.85E, 21.95N): 42.8° to 43.6°")
print("Geometry Conclusion: Swaths shift by several kilometers due to orbital baseline variation;")
print("geometry cannot be assumed identical solely because relative track is 19.")

In [ ]:
# Step 5: HDF5 Dataset Architecture (JPL D-102274 Inspection)
# Load verified dataset inventory
hdf5_file = Path("../data/processed/hdf5_dataset_inventory.json")

if hdf5_file.exists():
    inv = json.load(open(hdf5_file))
    print(f"Representative Product: {inv['representative_product']['granule_id']}")
    print(f"CRS: {inv['coordinate_reference_system']['crs_name']} (EPSG: {inv['coordinate_reference_system']['epsg_code']})\n")
    
    print("Verified Datasets under /science/LSAR/GCOV/grids/frequencyA:")
    for ds in inv["verified_datasets"]:
        dims = " x ".join(ds["dimensions"]) if isinstance(ds["dimensions"], list) else "scalar"
        print(f"  - {ds['dataset_path'].split('/')[-1]:<22} | Type: {ds['datatype']:<10} | Units: {ds['units']:<16} | Dims: {dims}")

In [ ]:
# Step 6: Remote Access Feasibility Audit (HTTP Range Requests)
# Testing whether HDF5 data can be sliced remotely over HTTP without downloading full files.

import urllib.request
import ssl

sample_h5_url = (
    "https://nisar.asf.earthdatacloud.nasa.gov/NISAR/NISAR_L2_GCOV_PROVISIONAL_V1/"
    "NISAR_L2_PR_GCOV_032_019_D_077_2005_QPDH_A_20261001T124313_20261001T124348_P05023_N_F_J_001/"
    "NISAR_L2_PR_GCOV_032_019_D_077_2005_QPDH_A_20261001T124313_20261001T124348_P05023_N_F_J_001.h5"
)

print("[INFO] Probing HTTP Range request support on NASA Earthdata Cloud...")
try:
    req = urllib.request.Request(sample_h5_url, headers={"User-Agent": "Mozilla/5.0", "Range": "bytes=0-512"})
    resp = urllib.request.urlopen(req, timeout=10)
    print(f"Response Status: {resp.status}")
    print(f"Content-Type: {resp.headers.get('Content-Type')}")
    if "text/html" in resp.headers.get("Content-Type", ""):
        print("[AUDIT RESULT] Endpoint redirected to Earthdata Login URS HTML page.")
        print("Conclusion: Unauthenticated HTTP range requests are blocked by NASA Earthdata.")
except Exception as e:
    print(f"[AUDIT RESULT] Remote probe: {e}")

print("\n[REMOTE HDF5 SLICING LIMITATIONS]")
print("1. Standard POSIX h5py cannot stream HDF5 slices over unauthenticated HTTP URLs.")
print("2. In NASA Earthdata Cloud, HTTP range requests require Bearer token/cookie chains.")
print("3. Each HDF5 chunk request incurs high network latency (~50-100ms per B-tree traverse).")
print("4. Solution: Download full granule to Google Colab cloud scratch storage, then slice locally in Colab.")

In [ ]:
# Step 7: Gosaba Subsetting Mathematics (EPSG:4326 to EPSG:32645)
# Geographic Bounds: 88.70E to 89.00E, 21.80N to 22.10N

subset_file = Path("../data/processed/gosaba_subset_metadata.json")
if subset_file.exists():
    meta = json.load(open(subset_file))
    utm_bbox = meta["projected_bounds_epsg_32645_utm45n"]["bounding_box_m"]
    print("=== GOSABA PROJECTED SUBSET BOUNDS (UTM 45N / EPSG:32645) ===")
    print(f"X (Easting):  {utm_bbox['min_x_easting']:,.2f} m to {utm_bbox['max_x_easting']:,.2f} m (Width: {utm_bbox['width_km']} km)")
    print(f"Y (Northing): {utm_bbox['min_y_northing']:,.2f} m to {utm_bbox['max_y_northing']:,.2f} m (Height: {utm_bbox['height_km']} km)")
    print(f"Total Area:   {utm_bbox['total_area_sq_km']} sq km\n")
    
    print("Memory & Dimension Reductions:")
    for res_name, res_info in meta["resolution_configurations"].items():
        print(f"  [{res_name}] Shape: {res_info['subset_rows']} rows x {res_info['subset_columns']} cols")
        print(f"    - Single Float32 Band: {res_info['uncompressed_float32_single_band_mb']} MB")
        print(f"    - 10-band Quad-Pol Stack: {res_info['uncompressed_10_band_quad_pol_stack_mb']} MB")
        print(f"    - Storage Savings: {res_info['compression_savings_vs_full_scene']}\n")

### Step 8: Spatial Resolution & Cross-Frequency Harmonization Plan

#### 1. Native Pixel Spacings
- **NISAR L-band (L-SAR)**: Native GCOV pixel spacing is **10 m** (or 20 m depending on bandwidth mode).
- **NISAR S-band (S-SAR)**: Native GCOV pixel spacing is **10 m** or **20 m** (typically 20 m over regional coastal tracks).

#### 2. Harmonization onto a Common Spatial Grid
1. **Master Reference Grid**:
   - Coordinate Reference System: WGS 84 / UTM Zone 45N (EPSG:32645).
   - Anchor: $X_{min} = 675,370	ext{ m}$, $Y_{max} = 2,445,260	ext{ m}$.
   - Nominal Target Resolution: **20 m $	imes$ 20 m**.
2. **Radiometric Preservation Kernel**:
   - Backscatter power layers must be resampled using **area-weighted multi-look averaging** or **bilinear interpolation on linear power**.
   - *Never use nearest neighbor on backscatter power* (induces severe blocky artifacts and drops speckle statistics).
   - *Never use cubic convolution or higher-order splines* (can introduce negative power overshoots on linear SAR power).
   - Discrete quality and validity masks must use **nearest neighbor**.
3. **Execution Guardrail**:
   - Cross-frequency pixel-to-pixel comparisons will not be performed until both datasets are independently ingested, subsetted, and verified against the master grid in Phase 3B.